# Lab 16: Vision AI Mini-Project: Object Counter and Missing-Item Alert

**Duration:** 4 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
You reach the end of the vision labs with a working AI application. You build **two small products** from the same detector. The first is an **object counter with alerts**: it counts one kind of object in the camera picture, and when the count reaches your limit it draws ALERT, saves a picture and adds a line to a log that opens in Excel. The second is a **missing-item alert** (a tool board or a desk check): it watches a zone, and when a required item is away for too long it raises an alarm, logs it and can send it to your phone. Both use a pretrained model, so you concentrate on the **decision rules**.

**By the end of this lab you can:**
1. Name the five steps of a vision application (capture, detect, decide, act, record) and find each one in the code.
2. Tune the confidence limit, the alert limit and the cooldown to balance false alerts and missed alerts.
3. Check which required items are inside a zone and decide when an item is missing, with a hold time for each item.
4. Log every event, open the log in Excel, and send an alarm to a phone with MQTT (optional).
5. Test the project in several situations and present it.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Privacy.** The camera sees faces, and the alert pictures are saved in your folder. Do not share them.

## 2. Background

**From lab work to an application.** Almost every vision application follows the same five steps. You learned each one in this course:

| Step | What happens | In this notebook | Learned in |
|---|---|---|---|
| Capture | Read a picture from the camera | the live view reads the camera | Lab 6 |
| Detect | Find the objects in it | `model(frame, ...)` in `process` | Labs 9 and 10 |
| Decide | Compare the count or the position with a rule | `count >= alert_at`, `items_on_board` | Lab 4 |
| Act | Show an alert and save a picture | `cv2.putText`, `save_picture` | Lab 6 |
| Record | Write a line in a log file | `log_alert`, `log_event` | Lab 4 |

You can also use the tracking of Lab 14 in your own idea (Part E) and the fast engine of Lab 15 (the project uses it if it is there).

**Good alerts.** An alert has two ways to fail. A *false alert* fires when nothing is wrong. A *missed alert* does not fire when something is. You tune three settings to balance them:

| Setting | If you raise it | If you lower it |
|---|---|---|
| `CONFIDENCE` | Fewer false alerts, but real objects are missed | More objects found, but more false alerts |
| `ALERT_AT` | Alerts only for bigger groups | Alerts sooner |
| `COOLDOWN` | Fewer pictures and log lines | More pictures and log lines |

The cooldown stops the program from saving a new picture on every frame while the alert is on. Without it the disk and the log would fill in seconds.

**Is an item on the board?** For each detection the program takes the **centre of the box**. If the centre is inside the zone (the board) and the confidence is high enough, the item is **on the board**. `cv2.pointPolygonTest` tells if a point is inside a polygon.

**The hold time, for every item.** Each required item has its own state, `OK` or `MISSING`. The state changes only after the new situation lasted `HOLD` seconds. If you pick up a tool for two seconds, nothing happens. If you take it away for longer, the alarm is raised. When the tool comes back, the program logs how long it was missing.

| | If you raise `HOLD` | If you lower `HOLD` |
|---|---|---|
| Effect | Fewer false alarms, but a late alarm | Fast alarms, but flicker when the model misses the item for a moment |

**MQTT (optional).** MQTT sends short messages between programs. Your Jetson has its own MQTT broker (see the Details of your student card on the teacher page). A phone app such as *IoT MQTT Panel* can subscribe to a topic and show the alarm on the phone.

**Change one thing at a time.** When a program stops working, you must know which change broke it. Change one thing, run it, then change the next. If you break the notebook, close it and run `import labview; labview.reset(16)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare and Plan (20 min)

### Step 1: Get the Model Files
The fast model (`yolo26n_fp16.engine`) is the one you can build in Lab 15. If it is missing, the program uses the plain `yolo26n.pt` and still works. This cell copies what exists into this folder.

In [ ]:
import glob
import os
import shutil

print("This notebook is saved in:", os.getcwd())

for source in ["../lab15/yolo26n_fp16.engine", "../lab15/yolo26n.pt", "../models/yolo26n.pt"]:
    name = os.path.basename(source)
    if os.path.exists(source) and not os.path.exists(name):
        shutil.copy(source, name)
        print("copied", source)

print("Model files here:", sorted(glob.glob("*.engine") + glob.glob("*.pt")))
if not os.path.exists("yolo26n_fp16.engine"):
    print("No FP16 engine here. The program will use yolo26n.pt (slower). Build the engine in Lab 15 to make it faster.")

### Step 2: Plan Your Two Projects
Pick one idea for each product, or invent your own. Both must use classes from the 80 classes of Lab 10 (for example person, cup, bottle, cell phone, scissors, car, cat, dog, bicycle).

| Counter idea | Class | Alert rule |
|---|---|---|
| Queue counter | person | Alert when 3 or more people are in view |
| Desk tidy check | bottle, cup | Alert when 4 or more items are on the desk |
| Phone reminder | cell phone | Alert when a phone is in view |
| Pet at the door | cat or dog | Alert when one is in view |

| Board idea | Board | Required items |
|---|---|---|
| Lab kit check | A tray | scissors, cup, cell phone |
| Desk check before you leave | The desk | bottle, cup, cell phone |
| Tool board | A sheet on the wall | scissors and other tools you can borrow |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Counter project name:
- What will it count, and when is the alert raised:
- Board project: the board and the required items:
- Who should get the alarm and what should they do:

---
# Part B: The Object Counter with Alerts (45 min)

### Step 1: The Settings
These are the only lines you must change to get a first working project. Change them to match your plan, then run the cell. Words in capital letters are settings.

In [ ]:
# ---------------- Settings you can change ----------------
TARGET = "person"        # the class to count (any of the 80 class names)
ALERT_AT = 3             # alert when this many or more are in view
CONFIDENCE = 0.40        # confidence limit
COOLDOWN = 5             # seconds between two alerts (picture and log line)
MODEL = "yolo26n_fp16.engine"
# ----------------------------------------------------------
print("Counting", TARGET, "| alert at", ALERT_AT)

### Step 2: Load the Model
The first time the engine loads, it can take a few seconds. If you see `not found`, the slower `.pt` model is used.

In [ ]:
import csv
import time
from datetime import datetime

import cv2
import torch
from ultralytics import YOLO

import labview

device = 0 if torch.cuda.is_available() else "cpu"
if not os.path.exists(MODEL):
    print(f"{MODEL} not found, using yolo26n.pt instead (slower)")
    MODEL = "yolo26n.pt"
model = YOLO(MODEL, task="detect")
print("Using model:", MODEL)

### Step 3: The Program
Read this cell before you run it. It has one function for each job:

| Function | What it does |
|---|---|
| `save_picture(img)` | Saves the picture of an alert as `alert_<date>_<time>.jpg` |
| `log_alert(count, picture)` | Adds one line to `alerts.csv` and rewrites `alerts.xlsx` (open it in Excel) |
| `process(frame, state)` | Runs the model on one picture, counts, decides and acts. The live view calls it for every picture. |

In [ ]:
run = {"last_alert": 0.0, "note": "", "warned": False}


def save_picture(img):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:-3]      # with milliseconds, so two alerts never share a name
    name = f"alert_{stamp}.jpg"
    cv2.imwrite(name, img)
    return name


def write_excel():
    """Rewrite alerts.xlsx from alerts.csv, so it opens in Excel with a header and a number column."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        if not run["warned"]:
            run["warned"] = True
            print("alerts.xlsx needs openpyxl. alerts.csv works in Excel too. Ask your instructor.")
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Alerts"
    with open("alerts.csv", newline="") as f:
        rows = list(csv.reader(f))
    for i, row in enumerate(rows):
        if i > 0:
            row[2] = int(row[2])
        sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABCD", (20, 12, 8, 32)):
        sheet.column_dimensions[column].width = width
    book.save("alerts.xlsx")


def log_alert(count, picture):
    """Add one line to the log files (alerts.csv and alerts.xlsx)."""
    first = not os.path.exists("alerts.csv")
    with open("alerts.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "class", "count", "picture"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), TARGET, count, picture])
    write_excel()


def process(frame, state):
    result = model(frame, conf=state["Confidence"], device=device, verbose=False)[0]
    count = sum(1 for c in result.boxes.cls if result.names[int(c)] == TARGET)
    shown = result.plot()

    alert = count >= int(state["Alert at"])
    colour = (0, 0, 255) if alert else (0, 255, 0)
    cv2.putText(shown, f"{TARGET}: {count}   {state['fps']:.0f} FPS", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, colour, 2)

    if alert:
        cv2.putText(shown, "ALERT", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
        now = time.time()
        if now - run["last_alert"] > COOLDOWN:
            run["last_alert"] = now
            picture = save_picture(shown)
            log_alert(count, picture)
            run["note"] = f"ALERT: {count} {TARGET}, saved {picture}"

    if run["note"]:
        cv2.putText(shown, run["note"][:70], (10, shown.shape[0] - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return shown


print("The program is ready.")

### Step 4: Run It
Run the cell. The camera picture appears under it with the count and the FPS. Use the sliders to change the confidence limit and the alert limit while it runs. Show your objects to the camera and watch the count. When the count reaches the alert limit, the word ALERT appears, a picture is saved and a line is added to the log. Press **Stop** when you are done.

The first time the engine loads, it can take a few seconds before the picture appears.

In [ ]:
run.update(last_alert=0.0, note="")
live = labview.live(process, controls={"Confidence": (0.05, 0.95, 0.05, CONFIDENCE),
                                       "Alert at": (1, 10, 1, ALERT_AT)})

### Step 5: See the Results
Run this cell after the alert has fired (and after you pressed Stop). It lists the alert pictures, shows the log and the last picture. The files are in the folder of this notebook: `alerts.csv`, `alerts.xlsx` and the pictures. To open the Excel file, use the file browser on the left of JupyterLab, right-click `alerts.xlsx` and choose **Download**, then open it on your laptop.

In [ ]:
from IPython.display import Image, display


def show_results():
    pictures = sorted(glob.glob("alert_*.jpg"))
    print("Alert pictures:", len(pictures))
    for name in ("alerts.csv", "alerts.xlsx"):
        print(f"{name:<12}", "found" if os.path.exists(name) else "not created yet (it appears at the first alert)")
    if os.path.exists("alerts.csv"):
        print()
        with open("alerts.csv") as f:
            print(f.read())
    if pictures:
        display(Image(filename=pictures[-1], width=420))


show_results()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Model file used (the output of Step 2):
- FPS shown on the screen:
- Did the alert fire when you expected?:
- First line of the log after the header:


---
# Part C: Test and Tune the Counter (25 min)
A real project must be tested. Delete the old log first with the cell below. Then try at least **four situations** and record the result.

In [ ]:
# Delete the old log and pictures to start a clean test
for name in glob.glob("alert_*.jpg") + ["alerts.csv", "alerts.xlsx"]:
    if os.path.exists(name):
        os.remove(name)
print("Clean. Files left:", sorted(glob.glob("alert*")))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1: normal view with the right objects. Expected / what happened:
- Test 2: objects partly hidden. Expected / what happened:
- Test 3: dim light. Expected / what happened:
- Test 4: a wrong object (not your class). Expected / what happened:


If you get false alerts, raise `CONFIDENCE`. If real objects are missed, lower it. Change it in steps of 0.05 with the slider.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- CONFIDENCE that worked best:
- Number of false alerts in two minutes of testing:
- Number of missed alerts in two minutes of testing:


---
# Part D: Build the Alarm (50 min)

### Step 1: The Settings
`REQUIRED` are the items that must be on the board. Use the class names of the 80 classes of Lab 10. `ZONE` is the board as a polygon of corners, as (x, y) fractions of the picture (like Lab 12). `HOLD` is the number of seconds an item must be absent (or back) before the state changes.

In [ ]:
# ---------------- Settings you can change ----------------
REQUIRED = ["cup", "cell phone"]     # the items that must be on the board (class names of the 80 classes of Lab 10)
ZONE = [(0.05, 0.10), (0.95, 0.10), (0.95, 0.95), (0.05, 0.95)]    # the board: corners as (x, y) fractions
HOLD = 3.0                    # seconds an item must be absent (or back) before the alarm or the all-clear
BOARD_CONFIDENCE = 0.50             # confidence limit
# ----------------------------------------------------------
print("Required:", REQUIRED, "| hold", HOLD, "s | model", MODEL)

In [ ]:
import numpy as np

unknown = [name for name in REQUIRED if name not in model.names.values()]
print("Classes of the model:", len(model.names))
if unknown:
    print("These required items are not a class of this model:", unknown, "-> check REQUIRED")
else:
    print("All required items are classes of the model.")

### Step 2: The Program
Read this cell before you run it.

| Function | What it does |
|---|---|
| `zone_polygon(shape)` | The `ZONE` in pixels for a picture of this size |
| `items_on_board(found, shape)` | From the detections (class name, box) it returns the set of required items whose box **centre** is inside the zone |
| `update_items(on_board, now)` | The hold rule for each required item. Returns a list of events `(item, "MISSING" or "BACK", seconds)` when a state changes. |
| `log_event(item, event, seconds)` | Adds a line to `board_log.csv` and rewrites `board_log.xlsx` |
| `process_board(frame, state)` | Detects, checks, raises the alarm and draws. The live view calls it for every picture. |

In [ ]:
items = {name: {"state": "OK", "since": time.time(), "cand": None, "cand_since": 0.0} for name in REQUIRED}
board = {"note": "", "warned": False, "client": None}


def zone_polygon(shape):
    h, w = shape[:2]
    return np.array([[int(x * w), int(y * h)] for x, y in ZONE], dtype=np.int32)


def items_on_board(found, shape):
    """found: a list of (class name, x1, y1, x2, y2). Returns the set of REQUIRED items whose box centre is inside the zone."""
    zone = zone_polygon(shape).reshape(-1, 1, 2)
    on_board = set()
    for name, x1, y1, x2, y2 in found:
        if name in REQUIRED and cv2.pointPolygonTest(zone, ((x1 + x2) / 2, (y1 + y2) / 2), False) >= 0:
            on_board.add(name)
    return on_board


def update_items(on_board, now):
    """The hold rule for each required item. Returns a list of (item, event, seconds in the old state)."""
    events = []
    for name, item in items.items():
        wanted = "OK" if name in on_board else "MISSING"
        if wanted == item["state"]:
            item["cand"] = None
        elif item["cand"] != wanted:
            item["cand"], item["cand_since"] = wanted, now
        elif now - item["cand_since"] >= HOLD:
            events.append((name, "MISSING" if wanted == "MISSING" else "BACK", now - item["since"]))
            item.update(state=wanted, since=now, cand=None)
    return events


def write_board_excel():
    """Rewrite board_log.xlsx from board_log.csv, so it opens in Excel with a header and number columns."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        if not board["warned"]:
            board["warned"] = True
            print("board_log.xlsx needs openpyxl. board_log.csv works in Excel too. Ask your instructor.")
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Board"
    with open("board_log.csv", newline="") as f:
        rows = list(csv.reader(f))
    for i, row in enumerate(rows):
        if i > 0:
            row[3] = float(row[3])
        sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABCD", (20, 14, 12, 26)):
        sheet.column_dimensions[column].width = width
    book.save("board_log.xlsx")


def log_event(item, event, seconds):
    first = not os.path.exists("board_log.csv")
    with open("board_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "item", "event", "seconds_in_previous_state"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), item, event, round(seconds, 1)])
    write_board_excel()


def send_alarm(item, event):
    """Optional: tell the phone. Works only after Part E Step 1 made board['client']."""
    if board["client"] is not None:
        try:
            board["client"].publish("board/alarm", f"{item} {event}")
        except Exception as error:
            print("Could not send to MQTT:", error)


def process_board(frame, state):
    result = model(frame, imgsz=640, conf=state["Confidence"], device=device, verbose=False)[0]
    shown = frame.copy()
    found = []
    for box, cls in zip(result.boxes.xyxy.tolist(), result.boxes.cls):
        x1, y1, x2, y2 = (int(v) for v in box)
        name = result.names[int(cls)]
        found.append((name, x1, y1, x2, y2))
        cv2.rectangle(shown, (x1, y1), (x2, y2), (0, 255, 0) if name in REQUIRED else (160, 160, 160), 2)
        cv2.putText(shown, name, (x1, max(y1 - 6, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

    now = time.time()
    for item, event, seconds in update_items(items_on_board(found, frame.shape), now):
        log_event(item, event, seconds)
        send_alarm(item, event)
        board["note"] = f"{item} {event} at {datetime.now().strftime('%H:%M:%S')}"

    missing = [name for name, item in items.items() if item["state"] == "MISSING"]
    colour = (0, 0, 255) if missing else (0, 200, 0)
    cv2.polylines(shown, [zone_polygon(frame.shape)], True, colour, 3)
    cv2.putText(shown, "ALARM: " + ", ".join(missing) + " missing" if missing else "All items on the board", (10, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.85, colour, 3)
    y = 62
    for name, item in items.items():
        text = f"{name}: {item['state']}"
        if item["cand"]:
            text += f"  (changing in {max(0.0, HOLD - (now - item['cand_since'])):.1f} s)"
        cv2.putText(shown, text, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255) if item["state"] == "MISSING" else (255, 255, 255), 2)
        y += 24
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    if board["note"]:
        cv2.putText(shown, board["note"], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    return shown


print("The program is ready.")

### Step 3: Check the Logic
Before the live test, check `items_on_board` and `update_items` with fake detections. Item `A` is inside the zone and item `B` is outside, so only `A` is on the board.

In [ ]:
saved_required, saved_items = list(REQUIRED), dict(items)
REQUIRED[:] = ["A", "B"]
items.clear()
items.update({n: {"state": "OK", "since": 0.0, "cand": None, "cand_since": 0.0} for n in REQUIRED})

shape = (480, 640, 3)
found = [("A", 300, 200, 340, 240), ("B", 0, 0, 20, 20)]            # B is in the corner, outside the zone
print("On the board:", items_on_board(found, shape), "(expected {'A'})")
print(update_items({"A"}, 10.0), "<- B starts to be missing: no event yet")
print(update_items({"A"}, 11.0), "<- still waiting")
print(update_items({"A"}, 13.5), "<- after HOLD seconds: B is MISSING")
print(update_items({"A", "B"}, 14.0), update_items({"A", "B"}, 17.5), "<- B is back after HOLD seconds")

REQUIRED[:] = saved_required                                         # put the real settings back
items.clear()
items.update(saved_items)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did all the lines give the expected result?:
- How long was B missing when the event was reported?:
- Why does the program need a separate state for each item?:


### Step 4: Run the Alarm
Run the cell. Put the required items on the board (inside the green outline). Take one away for longer than `HOLD` seconds: the outline turns red and `ALARM` shows with the item name. Put it back: after `HOLD` seconds the alarm clears. Take an item away for only one second: nothing should happen. Press **Stop** when you are done.

In [ ]:
now = time.time()
for item in items.values():
    item.update(state="OK", since=now, cand=None)
board["note"] = ""
live = labview.live(process_board, controls={"Confidence": (0.05, 0.95, 0.05, BOARD_CONFIDENCE)})

### Step 5: See the Results
Run this cell after you pressed Stop. To open the Excel file, use the file browser on the left, right-click `board_log.xlsx`, choose **Download** and open it on your laptop.

In [ ]:
def show_board_results():
    for name in ("board_log.csv", "board_log.xlsx"):
        print(f"{name:<16}", "found" if os.path.exists(name) else "not created yet (it appears at the first event)")
    if os.path.exists("board_log.csv"):
        print()
        with open("board_log.csv") as f:
            print(f.read())


show_board_results()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- FPS shown on the screen (compare with Part B):
- How many seconds did it take from taking an item away to the alarm? Does it match HOLD?:
- What happened when you took an item away for one second?:
- First line of the log after the header:


---
# Part E: Send the Alarm to Your Phone and Make It Your Own (35 min)

### Step 1: Send to the Phone with MQTT (optional)
Your Jetson has a personal MQTT broker. Open **Details** on your student card on the teacher page to see the host, the port, the user and the password. In the app *IoT MQTT Panel* on your phone, connect to that broker and add a **Text** (log) panel that subscribes to the topic `board/alarm`. Then run the cell. After that, every missing and returned event is also sent to the phone.

In [ ]:
try:
    board["client"] = labview.mqtt()
    print("Connected. Messages go to the topic board/alarm.")
    print("Broker details:", {k: v for k, v in labview.mqtt_info().items() if k != "password"})
    board["client"].publish("board/alarm", "board alarm ready")
except SystemExit as error:
    print(error)
except Exception as error:
    print("No MQTT connection:", error)

Run Step 4 of Part D again and take an item away. Look at the phone.

### Step 2: Make It Your Own
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the live cell again.

- Add or remove items in `REQUIRED`, or change the `ZONE` to the shape of your board.
- Show how long an item has been missing in minutes and seconds on the screen.
- Make a **count of alarms** per item and show it on the screen.
- Raise an alarm only when **two or more** items are missing, or when a **special** item is missing at once.
- Send a second message when everything is back on the board.
- Receive a message **from** the phone, for example `reset`, to clear the alarm. See `labview.mqtt(on_message, subscribe)` in Lab 16's notes.

> **Tip.** Change one thing at a time and run it after each change. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the phone get the messages? What did it show?:
- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


---
# Part F: Test and Present (30 min)

### Step 1: Test the Alarm
Delete the old log, run the alarm again and test each situation.

In [ ]:
for name in ("board_log.csv", "board_log.xlsx"):
    if os.path.exists(name):
        os.remove(name)
print("Clean. Files left:", sorted(glob.glob("board_log*")))

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | All items on the board for 10 seconds | No alarm | |
| 2 | One item taken away for 1 second | No alarm (shorter than `HOLD`) | |
| 3 | One item taken away for 10 seconds | ALARM after about `HOLD` seconds, one log line | |
| 4 | The item put back | Alarm clears after about `HOLD` seconds, one log line | |
| 5 | Two items taken away | ALARM with both names | |
| 6 | An item put **beside** the board (outside the zone) | ALARM (not on the board) | |
| 7 | An item that is not in `REQUIRED` put on the board | No change | |
| 8 | Dim light | Compare with test 1 | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 4: were all four as expected?:
- Test 5 and 6:
- Test 7 and 8:
- HOLD and CONFIDENCE that worked best:


### Step 2: Report and Present
Fill in the report. Then give a **3-minute demonstration**: show the live counter with its alert and picture, the live board with an alarm and the all-clear, the logs in Excel and the phone message.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Board idea and required items:
- Counter: class, alert limit and the FPS shown:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Congratulations.** You started this course by setting up your Jetson and you now have working AI vision applications on a small GPU board. The same steps, prepare the picture, run a model, act on the result and make it fast enough, are used in factories, shops and farms. Next you give the Jetson ears and a voice: speech to text, text to speech and keyword spotting.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The counter runs live, shows the count and the FPS, and raises the alert at your limit | |
| 2 | The alert saves a picture and adds a line to the log (alerts.csv / alerts.xlsx) | |
| 3 | The counter test in Part C is done, with the best CONFIDENCE written | |
| 4 | The logic check of the board (Part D Step 3) gives the expected results | |
| 5 | The live view shows the board outline and the state of each item | |
| 6 | Taking an item away for longer than `HOLD` raises the alarm, and a short absence does not | |
| 7 | Putting the item back clears the alarm and logs how long it was missing | |
| 8 | The board log is shown in Excel (`board_log.xlsx`) with at least 2 lines | |
| 9 | At least one improvement works and you explain the code you changed | |
| 10 | The test table is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. Name the five steps of a vision application and say which function or line of your program does each one.:
- 2. What is a false alert and what is a missed alert? Which setting changes them most?:
- 3. Why does the counter use a cooldown? What would happen without it?:
- 4. How does the program decide that an item is on the board? Why does it use the centre of the box?:
- 5. What does `HOLD` do? What happens if `HOLD` is 0?:
- 6. An item is taken away for 2 seconds and `HOLD` is 3. What does the program do? What if it is away for 10 seconds?:
- 7. Why does the board have a separate state for each required item?:
- 8. Which would you tune first if the alarm fires when nothing is wrong: CONFIDENCE or HOLD? Explain.:
- 9. Name one thing in this project that would be different if you used tracking (Lab 14).:
- 10. What is MQTT used for in this project? Name one other use.:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `yolo26n_fp16.engine not found, using yolo26n.pt instead` | The engine is not in this folder | Run Step 1 of Part A again. The engine is optional: build it in Lab 15 if you want more speed. |
| `Cannot open the camera` | Another program uses the camera, or the cable is loose | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| The count is always 0 | The class name in `TARGET` is wrong, or the confidence is too high | Check the spelling against the class names of Lab 10. Lower the confidence slider. |
| The alert fires all the time | `ALERT_AT` is too low, or the model finds things that are not there | Raise `ALERT_AT` or `CONFIDENCE` |
| No picture is saved | The cooldown has not passed yet, or the alert did not fire | Wait `COOLDOWN` seconds, check the count on the screen |
| `These required items are not a class of this model` | A name in `REQUIRED` is not one of the 80 classes | Use the exact class names of Lab 10, for example `cell phone` (with a space) |
| The alarm flickers | `HOLD` is too low or the confidence is too low | Raise `HOLD` and `BOARD_CONFIDENCE` a little |
| The outline never turns red | The items are inside the zone, or `REQUIRED` has items that are never seen | Move one item outside the green outline; check `REQUIRED` |
| `No MQTT connection` | The broker details are wrong or the phone app is not connected | Open Details on your student card on the teacher page and compare |
| `CUDA out of memory` | Other notebooks use the shared 8 GB | Press Stop, shut down other kernels (Kernel, Shut Down All Kernels) and run again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(16)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Ultralytics, Object counting: https://docs.ultralytics.com/guides/object-counting/
- OpenCV, pointPolygonTest: https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html
- MQTT: https://mqtt.org/